### Importing Libraries

In [75]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingRegressor
import joblib

### Loading the Data

In [39]:
df = pd.read_csv("final_productivity_dataset.csv")

In [40]:
df.head()

,study_hours_per_day,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,assignments_completed,attendance_percentage,focus_score,sleep_hours,stress_level,exercise_minutes,productivity_score,productivity_label
0,4.35,3.38,2.73,1.83,5.26,6,347,2,57.21,57,5.415,7.0,85.5,37.413481,Low
1,6.14,5.48,1.51,3.13,1.73,13,403,10,91.27,49,7.390,6.5,51.5,64.572195,Medium
2,4.98,4.83,3.63,0.18,4.71,1,419,8,63.14,38,4.680,5.0,96.0,33.910829,Low
3,3.19,10.06,3.95,5.75,2.52,9,178,18,40.51,50,5.340,6.0,35.0,44.919211,Low
4,7.67,3.02,1.59,5.46,5.65,8,436,7,45.53,41,6.155,7.0,67.5,71.967496,Medium


In [41]:
df.drop(columns = 'productivity_label', inplace = True)

In [42]:
df.head()

,study_hours_per_day,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,assignments_completed,attendance_percentage,focus_score,sleep_hours,stress_level,exercise_minutes,productivity_score
0,4.35,3.38,2.73,1.83,5.26,6,347,2,57.21,57,5.415,7.0,85.5,37.413481
1,6.14,5.48,1.51,3.13,1.73,13,403,10,91.27,49,7.390,6.5,51.5,64.572195
2,4.98,4.83,3.63,0.18,4.71,1,419,8,63.14,38,4.680,5.0,96.0,33.910829
3,3.19,10.06,3.95,5.75,2.52,9,178,18,40.51,50,5.340,6.0,35.0,44.919211
4,7.67,3.02,1.59,5.46,5.65,8,436,7,45.53,41,6.155,7.0,67.5,71.967496


In [43]:
df['productivity_score'] = round(df['productivity_score'], 2)

In [44]:
df.head()

,study_hours_per_day,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,assignments_completed,attendance_percentage,focus_score,sleep_hours,stress_level,exercise_minutes,productivity_score
0,4.35,3.38,2.73,1.83,5.26,6,347,2,57.21,57,5.415,7.0,85.5,37.41
1,6.14,5.48,1.51,3.13,1.73,13,403,10,91.27,49,7.390,6.5,51.5,64.57
2,4.98,4.83,3.63,0.18,4.71,1,419,8,63.14,38,4.680,5.0,96.0,33.91
3,3.19,10.06,3.95,5.75,2.52,9,178,18,40.51,50,5.340,6.0,35.0,44.92
4,7.67,3.02,1.59,5.46,5.65,8,436,7,45.53,41,6.155,7.0,67.5,71.97


In [45]:
x = df.drop('productivity_score', axis = 1)
y = df['productivity_score']

In [46]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [50]:
productive_features_train = x_train.copy()
productive_labels_train = y_train.copy()
num_attribs = productive_features_train.columns.tolist()

productive_features_test = x_test.copy()
productive_labels_test = y_test.copy()

In [51]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
])

In [52]:
productive_prepared_train = full_pipeline.fit_transform(productive_features_train)

In [71]:
rand_forest_model = GradientBoostingRegressor()
rand_forest_model.fit(productive_prepared_train, productive_labels_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [72]:
product_predict_train = rand_forest_model.predict(productive_prepared_train)

In [73]:
productive_r2_train = r2_score(productive_labels_train, product_predict_train)
productive_rmse_train = root_mean_squared_error(productive_labels_train, product_predict_train)
productive_mae_train = mean_absolute_error(productive_labels_train, product_predict_train)

productive_CV = cross_val_score(rand_forest_model, productive_prepared_train, productive_labels_train, scoring = "neg_root_mean_squared_error", cv = 10)

print(pd.Series(productive_CV).describe())
print(F"R2 Score:{productive_r2_train:.2f}")
print(f"RMSE Score:{productive_rmse_train:.2f}")
print(f"MAE Score:{productive_mae_train:.2f}")

count    10.000000
mean     -8.376154
std       0.157643
min      -8.577927
25%      -8.493883
50%      -8.392082
75%      -8.296370
max      -8.132225
dtype: float64
R2 Score:0.77
RMSE Score:7.87
MAE Score:6.29


In [74]:
productive_prepared_test = full_pipeline.transform(productive_features_test)
product_predict_test = rand_forest_model.predict(productive_prepared_test)
productive_r2_test = r2_score(productive_labels_test, product_predict_test)
productive_rmse_test = root_mean_squared_error(productive_labels_test, product_predict_test)
productive_mae_test = mean_absolute_error(productive_labels_test, product_predict_test)

print(F"R2 Score:{productive_r2_test:.2f}")
print(f"RMSE Score:{productive_rmse_test:.2f}")
print(f"MAE Score:{productive_mae_test:.2f}")

R2 Score:0.73
RMSE Score:8.27
MAE Score:6.63


### Exporting the Model

In [78]:
MODEL_FILE="model.pkl"
PIPELINE_FILE="pipeline.pkl"

def build_pipeline(num_attrbs):
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
    ])
    return full_pipeline

pipeline = build_pipeline(num_attribs)
joblib.dump(rand_forest_model, MODEL_FILE) 
joblib.dump(pipeline, PIPELINE_FILE) 
print("Model is Trained and Saved.")

Model is Trained and Saved.
